# 📚 Módulo 04 - Arquitecturas ML End-to-End

## 🎯 Objetivos

1. ✅ Comprender arquitecturas completas de ML
2. ✅ Integrar todos los componentes (datos, entrenamiento, serving, monitoreo)
3. ✅ Implementar pipelines de producción
4. ✅ Aplicar mejores prácticas de ingeniería

---

## 1️⃣ Arquitectura Completa

```
┌─────────────────────────────┐
│   DATA INGESTION          │
│   - APIs, DBs, Files       │
│   - Streaming / Batch      │
└────────────┬────────────────┘
             │
             ↓
┌────────────┬────────────────┐
│   DATA PROCESSING         │
│   🟪 Bronze → 🥈 Silver → 🥇 Gold  │
└────────────┬────────────────┘
             │
             ↓
┌────────────┬────────────────┐
│   FEATURE STORE           │
│   - Offline / Online       │
│   - Point-in-time correct  │
└────────────┬────────────────┘
             │
      ┌──────┼──────┐
      │            │
      ↓            ↓
┌──────┬──────┐   ┌──────┬──────┐
│   TRAINING    │   │   SERVING     │
│   - MLflow    │   │   - REST API  │
│   - CI/CD     │   │   - Batch     │
└──────┬──────┘   └──────┬──────┘
      │                    │
      └────────┬─────────┘
               │
               ↓
┌────────────┬────────────────┐
│   MONITORING              │
│   - Drift detection        │
│   - Performance tracking   │
│   - Alerting               │
└─────────────────────────────┘
```

---

## 2️⃣ Componentes Clave

### Feature Store

**¿Por qué usar Feature Store?**
* **Reutilización**: Features compartidas entre equipos
* **Consistencia**: Mismas features en train y serve
* **Performance**: Features precalculadas

```python
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Crear feature table
fe.create_table(
    name="main.ml.customer_features",
    primary_keys=["customer_id"],
    df=customer_features_df,
    description="Customer behavioral features"
)

# Usar en training
training_set = fe.create_training_set(
    df=labels_df,
    feature_lookups=[
        FeatureLookup(
            table_name="main.ml.customer_features",
            lookup_key="customer_id"
        )
    ],
    label="churn"
)
```

### Model Registry

```python
import mlflow

# Registrar modelo
mlflow.register_model(
    model_uri="runs:/abc123/model",
    name="churn_predictor"
)

# Promover a producción
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name="churn_predictor",
    version=5,
    stage="Production"
)
```

### Model Serving

**REST API automático**:

```python
# Databricks genera endpoint automáticamente
url = "https://<workspace>.cloud.databricks.com/serving-endpoints/churn_predictor/invocations"

headers = {"Authorization": f"Bearer {token}"}
data = {"dataframe_records": [{"customer_id": 123, "tenure": 24}]}

response = requests.post(url, headers=headers, json=data)
prediction = response.json()["predictions"][0]
```

---

## 3️⃣ Patrones de Despliegue

### Batch Scoring

```python
# Databricks Job diario
model = mlflow.pyfunc.load_model("models:/churn_predictor/Production")

new_customers = spark.table("gold.customers_to_score")
predictions = model.predict(new_customers.toPandas())

# Guardar predicciones
predictions_df.write.format("delta").mode("overwrite").saveAsTable("gold.churn_predictions")
```

### Real-Time Serving

```python
# Model Serving endpoint con autoscaling
from databricks import serving

serving.create_endpoint(
    name="churn_api",
    model_name="churn_predictor",
    model_version="5",
    workload_size="Small",  # Small, Medium, Large
    scale_to_zero_enabled=True
)
```

### Streaming

```python
# Spark Structured Streaming
stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "...")
    .load()
)

# Aplicar modelo
model_udf = mlflow.pyfunc.spark_udf(spark, "models:/churn_predictor/Production")
predictions = stream.withColumn("prediction", model_udf(struct([stream[x] for x in features])))

# Escribir a Delta
predictions.writeStream.format("delta").outputMode("append").start("/predictions")
```

---

## 4️⃣ Mejores Prácticas

### ✅ Separación de Ambientes

```
Dev       →  Desarrollo local, experimentos
Staging   →  Pre-producción, validaciones
Production →  Tráfico real
```

### ✅ Versionado Completo

* Código: Git
* Datos: Delta Lake time travel
* Features: Feature Store
* Modelos: MLflow Registry
* Infraestructura: Terraform

### ✅ Observabilidad

```python
# Logging estructurado
import logging

logger.info(
    "Prediction made",
    extra={
        "model_version": "5",
        "customer_id": 123,
        "prediction": 0.8,
        "latency_ms": 45
    }
)
```

### ✅ Disaster Recovery

* Backups automáticos de Delta Tables
* Modelos archivados en S3
* Plan de rollback documentado
* Runbooks para incidentes

---

## 5️⃣ Ejemplo: Pipeline Completo

```python
# pipeline.py

class MLPipeline:
    def __init__(self, config):
        self.config = config
        self.fe_client = FeatureEngineeringClient()
    
    def run(self):
        # 1. Ingerir datos
        raw_data = self.ingest_data()
        
        # 2. Procesar (Bronze → Silver → Gold)
        clean_data = self.clean_data(raw_data)
        features = self.engineer_features(clean_data)
        
        # 3. Guardar en Feature Store
        self.fe_client.write_table(
            name="main.ml.features",
            df=features,
            mode="merge"
        )
        
        # 4. Entrenar modelo
        training_set = self.create_training_set()
        model = self.train_model(training_set)
        
        # 5. Validar
        if self.validate_model(model):
            # 6. Registrar en MLflow
            model_uri = self.register_model(model)
            
            # 7. Desplegar a Staging
            self.deploy_to_staging(model_uri)
            
            # 8. A/B test
            if self.run_ab_test():
                # 9. Promover a Producción
                self.promote_to_production(model_uri)
        
        # 10. Monitorear
        self.setup_monitoring()
```

---

## ✅ Checklist de Proyecto Integrador

☑️ Arquitectura Medallion implementada  
☑️ Feature Store configurado  
☑️ Pipeline de entrenamiento automatizado  
☑️ CI/CD con tests  
☑️ Model Registry con stages  
☑️ Endpoint de serving  
☑️ Monitoreo de drift  
☑️ Alertas configuradas  
☑️ Documentación completa  
☑️ Plan de rollback  

---

**Universidad del Aconcagua 🇦🇷**